# Artificial Neural Network

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Importing the libraries

In [2]:
import numpy as np
import pandas as pd
import torch


In [ ]:
torch.__version__

'2.9.0+cpu'

## Part 1 - Data Preprocessing

### Importing the dataset

In [4]:
dataset=pd.read_csv('/content/drive/MyDrive/datafiles/Churn_Modelling.csv')
X=dataset.iloc[:,3:-1].values
y=dataset.iloc[:,-1].values

In [ ]:
dataset

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [ ]:

X

array([[619, 'France', 'Female', ..., 1, 1, 101348.88],
       [608, 'Spain', 'Female', ..., 0, 1, 112542.58],
       [502, 'France', 'Female', ..., 1, 0, 113931.57],
       ...,
       [709, 'France', 'Female', ..., 0, 1, 42085.58],
       [772, 'Germany', 'Male', ..., 1, 0, 92888.52],
       [792, 'France', 'Female', ..., 1, 0, 38190.78]], dtype=object)

In [ ]:
y

array([1, 0, 1, ..., 1, 1, 0])

### Encoding categorical data

*Label* Encoding the "Gender" column

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
X[:,2]=le.fit_transform(X[:,2])

In [ ]:
X

array([[619, 'France', 0, ..., 1, 1, 101348.88],
       [608, 'Spain', 0, ..., 0, 1, 112542.58],
       [502, 'France', 0, ..., 1, 0, 113931.57],
       ...,
       [709, 'France', 0, ..., 0, 1, 42085.58],
       [772, 'Germany', 1, ..., 1, 0, 92888.52],
       [792, 'France', 0, ..., 1, 0, 38190.78]], dtype=object)

One Hot Encoding the "Geography" column

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct=ColumnTransformer(transformers=[('encoder',OneHotEncoder(),[1])],remainder='passthrough')
X = np.array(ct.fit_transform(X))



In [ ]:
X

array([[1.0, 0.0, 0.0, ..., 1, 1, 101348.88],
       [0.0, 0.0, 1.0, ..., 0, 1, 112542.58],
       [1.0, 0.0, 0.0, ..., 1, 0, 113931.57],
       ...,
       [1.0, 0.0, 0.0, ..., 0, 1, 42085.58],
       [0.0, 1.0, 0.0, ..., 1, 0, 92888.52],
       [1.0, 0.0, 0.0, ..., 1, 0, 38190.78]], dtype=object)

### Splitting the dataset into the Training set and Test set

In [ ]:
# NumPy defaults to the object dtype if your original Pandas DataFrame contains mixed data types, hidden strings, or NaN (missing) values

print(X.dtype)

object


PyTorch needs to know exactly how many bits each number takes up to move data to the GPU. An object array is just a collection of pointers to Python objects scattered in memory, whereas float32 is a dense block of 32-bit numbers that PyTorch can process efficiently.

In [ ]:
# PyTorch's from_numpy() or torch.tensor() cannot convert object arrays
# because they require a contiguous block of numeric memory.

# Convert to float32 (the standard for PyTorch weights)
X = X.astype(np.float32)
y = y.astype(np.float32)

print(X.dtype)
print(y.dtype)


float32
float32


In [ ]:
# Turn data into tensors
# Otherwise this causes issues with computations later on
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

# View the first five samples
X[:5], y[:5]

(tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 6.1900e+02, 0.0000e+00, 4.2000e+01,
          2.0000e+00, 0.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0135e+05],
         [0.0000e+00, 0.0000e+00, 1.0000e+00, 6.0800e+02, 0.0000e+00, 4.1000e+01,
          1.0000e+00, 8.3808e+04, 1.0000e+00, 0.0000e+00, 1.0000e+00, 1.1254e+05],
         [1.0000e+00, 0.0000e+00, 0.0000e+00, 5.0200e+02, 0.0000e+00, 4.2000e+01,
          8.0000e+00, 1.5966e+05, 3.0000e+00, 1.0000e+00, 0.0000e+00, 1.1393e+05],
         [1.0000e+00, 0.0000e+00, 0.0000e+00, 6.9900e+02, 0.0000e+00, 3.9000e+01,
          1.0000e+00, 0.0000e+00, 2.0000e+00, 0.0000e+00, 0.0000e+00, 9.3827e+04],
         [0.0000e+00, 0.0000e+00, 1.0000e+00, 8.5000e+02, 0.0000e+00, 4.3000e+01,
          2.0000e+00, 1.2551e+05, 1.0000e+00, 1.0000e+00, 1.0000e+00, 7.9084e+04]]),
 tensor([1., 0., 1., 0., 0.]))

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

### Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

Check the Shapes

In [ ]:
X_train.shape,y_train.shape,X_test.shape,y_test.shape

((8000, 12), torch.Size([8000]), (2000, 12), torch.Size([2000]))

## Part 2 - Building the ANN

### Initializing the ANN

In [ ]:
# Standard PyTorch imports
from torch import nn

# Make device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [ ]:
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        super(BinaryClassifier, self).__init__()
        # 2. Create 2 nn.Linear layers capable of handling X and y input and output shapes
        self.layer1 = nn.Linear(input_dim, 24) # takes in input_dim features (X), produces 24 features(Neuron)
        self.layer2 = nn.Linear(24, 12) # takes in 24 features, produces 12 feature (y)
        self.output = nn.Linear(12, 1) # Output layer
        self.relu = nn.ReLU()


    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.output(x) # Added the output layer
        return x

model = BinaryClassifier(X_train.shape[1]).to(device)

### Preparing data for training





In [ ]:
from torch.utils.data import DataLoader, TensorDataset



# Step A: Ensure data is in Float32 Tensors (Standard for PyTorch)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

# Put data to target device
X_train_tensor, y_train_tensor = X_train_tensor.to(device), y_train_tensor.to(device)

# Step B: Combine X and y into one Dataset object
# This keeps your features and labels "glued" together during shuffling
train_data = TensorDataset(X_train_tensor, y_train_tensor)

# Step C: Create the DataLoader
# batch_size=32 is a standard starting point for most ANNs
train_loader = DataLoader(dataset=train_data, batch_size=32, shuffle=True)


/tmp/ipython-input-1615863743.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train_tensor = torch.tensor(y_train, dtype=torch.float32)


### Adding the second hidden layer

### Adding the output layer

## Part 3 - Training the ANN

### Training the ANN on the Training set

In [ ]:
import torch

# 1. Setup Loss and Optimizer
loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

def train_model(model, train_loader, epochs=100):
    for epoch in range(epochs):
        model.train() # Set to training mode

        train_loss, train_acc = 0, 0

        for X_batch, y_batch in train_loader:
            # Squeeze to ensure shapes are 1D: [batch_size]
            y_logits = model(X_batch).squeeze()
            y_batch = y_batch.squeeze()

            #1. Calculate the loss
            # The model's outputs (predictions/logits) are compared to the
            # ground truth and evaluated to see how wrong they are.
            loss = loss_fn(y_logits, y_batch)

            # 2. Accuracy Calculation (Manual)
            # Step A: Convert logits -> probabilities (0 to 1)
            y_probs = torch.sigmoid(y_logits)
            # Step B: Round probabilities -> classes (0 or 1)
            y_preds = torch.round(y_probs)
            # Step C: Compare to labels and calculate mean
            acc = (y_preds == y_batch).sum().item() / len(y_batch)

            # 3. Optimization steps

            # Zero gradients
            # The optimizers gradients are set to zero (they are accumulated by default)
            # so they can be recalculated for the specific training step.
            optimizer.zero_grad()

             # Backpropagation
             # Computes the gradient of the loss with respect for every model parameter
             # to be updated. This is known as backpropagation.
            loss.backward()


            # Update weights
            # Update the parameters with requires_grad=True with respect to the
            # loss gradients in order to improve them.
            optimizer.step()

            train_loss += loss.item()
            train_acc += acc

        # Calculate average train loss and accuracy
        avg_train_loss = train_loss / len(train_loader)
        avg_train_acc = (train_acc / len(train_loader)) * 100

        # Print results every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3} | Train Loss: {avg_train_loss:.4f} | Train Accuracy: {avg_train_acc:.2f}%")

# Run the training
train_model(model, train_loader)

Epoch  10 | Train Loss: 0.3429 | Train Accuracy: 85.97%
Epoch  20 | Train Loss: 0.3336 | Train Accuracy: 86.11%
Epoch  30 | Train Loss: 0.3282 | Train Accuracy: 86.44%
Epoch  40 | Train Loss: 0.3225 | Train Accuracy: 86.69%
Epoch  50 | Train Loss: 0.3175 | Train Accuracy: 86.74%
Epoch  60 | Train Loss: 0.3186 | Train Accuracy: 86.90%
Epoch  70 | Train Loss: 0.3112 | Train Accuracy: 86.96%
Epoch  80 | Train Loss: 0.3102 | Train Accuracy: 87.26%
Epoch  90 | Train Loss: 0.3083 | Train Accuracy: 87.34%
Epoch 100 | Train Loss: 0.3016 | Train Accuracy: 87.59%


## Part 4 - Making the predictions and evaluating the model

### Evaluation on test data set

In [ ]:
def evaluate_model(model, test_loader):
    model.eval() # Set to evaluation mode
    test_loss, test_acc = 0, 0
    with torch.inference_mode(): # Turn off gradient tracking for evaluation
        for X_test_batch, y_test_batch in test_loader:
            y_test_logits = model(X_test_batch).squeeze()
            y_test_batch = y_test_batch.squeeze()

            loss = loss_fn(y_test_logits, y_test_batch)
            y_test_probs = torch.sigmoid(y_test_logits)
            y_test_preds = torch.round(y_test_probs)
            acc = (y_test_preds == y_test_batch).sum().item() / len(y_test_batch)

            test_loss += loss.item()
            test_acc += acc

    # Calculate average test loss and accuracy
    avg_test_loss = test_loss / len(test_loader)
    avg_test_acc = (test_acc / len(test_loader)) * 100
    print(f"\nTest Loss: {avg_test_loss:.4f} | Test Accuracy: {avg_test_acc:.2f}%")

# Call the evaluate_model function after training

# Define test_loader
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

X_test_tensor, y_test_tensor = X_test_tensor.to(device), y_test_tensor.to(device)

test_data = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(dataset=test_data, batch_size=32, shuffle=False)

evaluate_model(model, test_loader)


Test Loss: 0.3806 | Test Accuracy: 85.47%


/tmp/ipython-input-1947384657.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_test_tensor = torch.tensor(y_test, dtype=torch.float32)


### Predicting the result of a single observation

**Homework**

Use our ANN model to predict if the customer with the following informations will leave the bank:

Geography: France

Credit Score: 600

Gender: Male

Age: 40 years old

Tenure: 3 years

Balance: \$ 60000

Number of Products: 2

Does this customer have a credit card? Yes

Is this customer an Active Member: Yes

Estimated Salary: \$ 50000

So, should we say goodbye to that customer?

**Solution**

In [ ]:
new_customer = np.array([[1.0, 0.0, 0.0, 600, 1, 40, 3, 60000, 2, 1, 1, 50000]], dtype=np.float32)

# Scale the new customer data using the fitted StandardScaler
new_customer_scaled = sc.transform(new_customer)

# Convert to PyTorch tensor
new_customer_tensor = torch.tensor(new_customer_scaled, dtype=torch.float32).to(device)

# Make prediction
model.eval()
with torch.inference_mode():
    # Get logits
    prediction_logits = model(new_customer_tensor)
    # Apply sigmoid to get probability
    prediction_prob = torch.sigmoid(prediction_logits)
    # Round to get binary prediction
    prediction = torch.round(prediction_prob)

if prediction.item() == 1:
    print("This customer is predicted to leave the bank.")
else:
    print("This customer is predicted to stay in the bank.")

This customer is predicted to stay in the bank.


Therefore, our ANN model predicts that this customer stays in the bank!

**Important note 1:** Notice that the values of the features were all input in a double pair of square brackets. That's because the "predict" method always expects a 2D array as the format of its inputs. And putting our values into a double pair of square brackets makes the input exactly a 2D array.

**Important note 2:** Notice also that the "France" country was not input as a string in the last column but as "1, 0, 0" in the first three columns. That's because of course the predict method expects the one-hot-encoded values of the state, and as we see in the first row of the matrix of features X, "France" was encoded as "1, 0, 0". And be careful to include these values in the first three columns, because the dummy variables are always created in the first columns.

### Predicting the Test set results

In [ ]:
model.eval() # Set the model to evaluation mode

with torch.inference_mode(): # Disable gradient calculations
    y_logits_test = model(X_test_tensor).squeeze() # Make predictions and squeeze the output
    y_pred_probs = torch.sigmoid(y_logits_test) # Apply sigmoid to get probabilities
    y_pred = torch.round(y_pred_probs) # Round to get binary predictions (0 or 1)

y_pred_numpy = y_pred.cpu().numpy() # Convert to numpy array for concatenation
y_test_numpy = y_test_tensor.cpu().numpy() # Convert y_test to numpy array

print(np.concatenate((y_pred_numpy.reshape(len(y_pred_numpy), 1), y_test_numpy.reshape(len(y_test_numpy), 1)), 1))

[[1. 0.]
 [0. 1.]
 [0. 0.]
 ...
 [0. 0.]
 [0. 0.]
 [0. 0.]]


### Making the Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test_numpy, y_pred_numpy)
print(cm)
accuracy_score(y_test_numpy, y_pred_numpy)

[[1520   75]
 [ 217  188]]


0.854